In [2]:
import pandas as pd
import numpy as np
import sqlite3

### TASK1  DATA INGESTION

In [ ]:

import pandas as pd

sales_df= pd.read_csv("sales_data.csv")
products_df = pd.read_csv("products.csv")
stores_df = pd.read_csv("stores.csv")

In [6]:
print(sales_df.shape)
print(sales_df.head())
print(sales_df.isnull().sum())

(15, 6)
   sale_id  store_id  product_id  quantity   sale_date   amount
0        1       101           1       2.0  2024-01-05  64000.0
1        2       102           3       5.0  2024-01-05   2250.0
2        3       103           2       1.0  2024-01-06   4500.0
3        4       104           5       3.0  2024-01-06   2697.0
4        5       105           4       2.0  2024-01-07   5998.0
sale_id       0
store_id      0
product_id    0
quantity      2
sale_date     0
amount        2
dtype: int64


In [8]:
print(products_df.shape)
print(products_df.head())
print(products_df.isnull().sum())

(12, 4)
   product_id     product_name     category  price
0           1       Samsung TV  Electronics  32000
1           2       Nike Shoes     Clothing   4500
2           3     Basmati Rice      Grocery    450
3           4  OnePlus Earbuds  Electronics   2999
4           5     Cotton Kurta     Clothing    899
product_id      0
product_name    0
category        0
price           0
dtype: int64


In [9]:
print(stores_df.shape)
print(stores_df.head())
print(stores_df.isnull().sum())

(12, 4)
   store_id              store_name       city region
0       101       RetailMart Bandra     Mumbai   West
1       102           RetailMart CP      Delhi  North
2       103  RetailMart Koramangala  Bangalore  South
3       104    RetailMart Salt Lake    Kolkata   East
4       105   RetailMart Anna Nagar    Chennai  South
store_id      0
store_name    0
city          0
region        0
dtype: int64


### TASK2 DATA CLEANING


In [5]:
duplicates_count = sales_df.duplicated().sum()
print(f"Found {duplicates_count} duplicate rows. Removing...")

sales_df = sales_df.drop_duplicates()
print("Shape after removing duplicates:", sales_df.shape)

Found 3 duplicate rows. Removing...
Shape after removing duplicates: (15, 6)


In [8]:
sales_df['quantity'] = sales_df['quantity'].fillna(0)

rows_before = sales_df.shape[0]
sales_df = sales_df.dropna(subset=['amount'])
rows_after = sales_df.shape[0]

print(f"Dropped {rows_before - rows_after} rows where amount was NULL")
print("Shape after cleaning nulls:", sales_df.shape)

Dropped 0 rows where amount was NULL
Shape after cleaning nulls: (13, 6)


In [32]:

sales_df['sale_date'] = pd.to_datetime(sales_df['sale_date'])
sales_df['amount']    = sales_df['amount'].astype(float)

print("\nData types after conversion:")
print(sales_df.dtypes)


Data types after conversion:
sale_id         int64
store_id        int64
product_id      int64
quantity      float64
sale_date      object
amount        float64
dtype: object


### TASK3 DATA TRANSFORMATION

In [27]:

merged_df = sales_df.merge(products_df, on='product_id', how='inner')
merged_df = merged_df.merge(stores_df, on='store_id', how='inner')

print("Final Merged DataFrame:")
print(merged_df)

Final Merged DataFrame:
    sale_id  store_id  product_id  quantity   sale_date   amount  \
0         1       101           1       2.0  2024-01-05  64000.0   
1         2       102           3       5.0  2024-01-05   2250.0   
2         3       103           2       1.0  2024-01-06   4500.0   
3         4       104           5       3.0  2024-01-06   2697.0   
4         5       105           4       2.0  2024-01-07   5998.0   
5         6       106           6       4.0  2024-01-07   1120.0   
6         7       107           7       1.0  2024-01-08   1299.0   
7         8       108           8       2.0  2024-01-08   3198.0   
8        10       110          10       1.0  2024-01-09   2499.0   
9        11       111           3       3.0  2024-01-10   1350.0   
10       12       112           1       1.0  2024-01-10  32000.0   
11       14       102           2       2.0  2024-01-11   9000.0   
12       15       103           6       5.0  2024-01-12   1400.0   

         product_name  

In [ ]:
#Adding total_revenue column
merged_df['total_revenue'] = merged_df['quantity'] * merged_df['price']

print("\ntotal_revenue stats:")
print("Mean :", np.mean(merged_df['total_revenue']))
print("Max  :", np.max(merged_df['total_revenue']))
print("Min  :", np.min(merged_df['total_revenue']))


total_revenue stats:
Mean : 10100.846153846154
Max  : 64000.0
Min  : 1120.0


In [ ]:
#total revenue per city
city_revenue = merged_df.groupby('city')['total_revenue'].sum()
city_revenue = city_revenue.sort_values(ascending=False)

print("\nTotal Revenue by City:")
print(city_revenue)


Total Revenue by City:
city
Mumbai        66499.0
Delhi         43250.0
Bangalore      7250.0
Chennai        5998.0
Chandigarh     3198.0
Kolkata        2697.0
Pune           1299.0
Hyderabad      1120.0
Name: total_revenue, dtype: float64


### TASK4 DATALOADING(SQLITE)

In [13]:
merged_df.to_csv('merged.csv', index=False)   

In [15]:
import sqlite3

conn = sqlite3.connect("retailmart.db")

merged_df.to_sql("retail_sales", conn, if_exists="replace", index=False)
print("data loaded successfully")
conn.close()

data loaded successfully


In [16]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("retailmart.db")

tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

print(tables)

conn.close()

           name
0  retail_sales


In [31]:
conn = sqlite3.connect("retailmart.db")

df = pd.read_sql(
    "SELECT * FROM retail_sales LIMIT 5;",
    conn
)

print(df)

conn.close()

   sale_id  store_id  product_id  quantity            sale_date   amount  \
0        1       101           1       2.0  2024-01-05 00:00:00  64000.0   
1        2       102           3       5.0  2024-01-05 00:00:00   2250.0   
2        3       103           2       1.0  2024-01-06 00:00:00   4500.0   
3        4       104           5       3.0  2024-01-06 00:00:00   2697.0   
4        5       105           4       2.0  2024-01-07 00:00:00   5998.0   

      product_name     category  price              store_name       city  \
0       Samsung TV  Electronics  32000       RetailMart Bandra     Mumbai   
1     Basmati Rice      Grocery    450           RetailMart CP      Delhi   
2       Nike Shoes     Clothing   4500  RetailMart Koramangala  Bangalore   
3     Cotton Kurta     Clothing    899    RetailMart Salt Lake    Kolkata   
4  OnePlus Earbuds  Electronics   2999   RetailMart Anna Nagar    Chennai   

  region  total_revenue  
0   West        64000.0  
1  North         2250.0  
2 

In [ ]:
conn = sqlite3.connect("retailmart.db")

df = pd.read_sql(
    "SELECT  product_name,sum(quantity) as total_qty FROM retail_sales GROUP by product_name ORDER BY total_qty desc limit 3 " ,
    conn)

print(df)

conn.close()

    product_name  total_qty
0  Sunflower Oil        9.0
1   Basmati Rice        8.0
2     Samsung TV        3.0


In [ ]:
conn = sqlite3.connect("retailmart.db")

df = pd.read_sql(
    "SELECT * FROM retail_sales LIMIT 5;",
    conn
)

print(df)

conn.close()

### TASK5 REPORTING AND INSIGHTS

In [ ]:
conn = sqlite3.connect("retailmart.db")

query = """
SELECT store_name,sale_date,SUM(total_revenue) AS daily_revenue FROM retail_sales
GROUP BY store_name, sale_date
ORDER BY sale_date;
"""

store_revenue = pd.read_sql(query, conn)

print(store_revenue)

conn.close()

                    store_name            sale_date  daily_revenue
0            RetailMart Bandra  2024-01-05 00:00:00        64000.0
1                RetailMart CP  2024-01-05 00:00:00         2250.0
2       RetailMart Koramangala  2024-01-06 00:00:00         4500.0
3         RetailMart Salt Lake  2024-01-06 00:00:00         2697.0
4        RetailMart Anna Nagar  2024-01-07 00:00:00         5998.0
5     RetailMart Jubilee Hills  2024-01-07 00:00:00         1120.0
6           RetailMart MG Road  2024-01-08 00:00:00         1299.0
7         RetailMart Sector 17  2024-01-08 00:00:00         3198.0
8      RetailMart Linking Road  2024-01-09 00:00:00         2499.0
9   RetailMart Connaught Place  2024-01-10 00:00:00        32000.0
10      RetailMart Indiranagar  2024-01-10 00:00:00         1350.0
11               RetailMart CP  2024-01-11 00:00:00         9000.0
12      RetailMart Koramangala  2024-01-12 00:00:00         1400.0


In [50]:
print("RETAILMART SUMMARY REPORT")
print("-"*40)

# Total merged_df
total_transactions = len(merged_df)

# Total revenue

total_revenue = (merged_df['quantity'] * merged_df['price']).sum()

# Top selling city
top_city = (
    merged_df.groupby("city")["total_revenue"]
    .sum()
    .idxmax()
)

# Top selling product
top_product = (
    merged_df.groupby("product_name")["quantity"]
    .sum()
    .idxmax()
)

print(f"Total Transactions : {total_transactions}")
print(f"Total Revenue      : ₹{total_revenue:,.2f}")
print(f"Top Selling City   : {top_city}")
print(f"Top Selling Product: {top_product}")

RETAILMART SUMMARY REPORT
----------------------------------------
Total Transactions : 13
Total Revenue      : ₹131,311.00
Top Selling City   : Mumbai
Top Selling Product: Sunflower Oil


### TASK6 PIPELINE & ERRORHANDLING

In [ ]:
import pandas as pd
import numpy as np
import sqlite3


def run_pipeline():

    try:

# Load
        sales_df = pd.read_csv("sales_data.csv")
        products_df = pd.read_csv("products.csv")
        stores_df = pd.read_csv("stores.csv")

# Clean
        sales_df = sales_df.drop_duplicates()

        sales_df["quantity"] = (
            sales_df["quantity"]
            .fillna(0)
        )

        sales_df = sales_df.dropna(
            subset=["amount"]
        )

        sales_df["sale_date"] = pd.to_datetime(
            sales_df["sale_date"]
        )

        sales_df["amount"] = (
            sales_df["amount"]
            .astype(float)
        )

        # Transform
        final_df = pd.merge(
            sales_df,
            products_df,
            on="product_id",
            how="left"
        )

        final_df = pd.merge(
            final_df,
            stores_df,
            on="store_id",
            how="left"
        )

        final_df["total_revenue"] = (
            final_df["quantity"]
            * final_df["price"]
        )

#sqlite
        conn = sqlite3.connect(
            "retailmart.db"
        )

        final_df.to_sql(
            "retail_sales",
            conn,
            if_exists="replace",
            index=False
        )

        conn.close()

        print(
            f"{len(final_df)} records loaded into retail_sales"
        )

        print(
            "Pipeline executed successfully!"
        )

        return final_df

    except FileNotFoundError as e:

        print(
            f"Missing file: {e}"
        )

    except Exception as e:

        print(
            f"Unexpected error: {e}"
        )

In [48]:
run_pipeline()

13 records loaded into retail_sales
Pipeline executed successfully!


,sale_id,store_id,product_id,quantity,sale_date,amount,product_name,category,price,store_name,city,region,total_revenue
0,1,101,1,2.0,2024-01-05,64000.0,Samsung TV,Electronics,32000,RetailMart Bandra,Mumbai,West,64000.0
1,2,102,3,5.0,2024-01-05,2250.0,Basmati Rice,Grocery,450,RetailMart CP,Delhi,North,2250.0
2,3,103,2,1.0,2024-01-06,4500.0,Nike Shoes,Clothing,4500,RetailMart Koramangala,Bangalore,South,4500.0
3,4,104,5,3.0,2024-01-06,2697.0,Cotton Kurta,Clothing,899,RetailMart Salt Lake,Kolkata,East,2697.0
4,5,105,4,2.0,2024-01-07,5998.0,OnePlus Earbuds,Electronics,2999,RetailMart Anna Nagar,Chennai,South,5998.0
5,6,106,6,4.0,2024-01-07,1120.0,Sunflower Oil,Grocery,280,RetailMart Jubilee Hills,Hyderabad,South,1120.0
6,7,107,7,1.0,2024-01-08,1299.0,Laptop Stand,Electronics,1299,RetailMart MG Road,Pune,West,1299.0
7,8,108,8,2.0,2024-01-08,3198.0,Denim Jeans,Clothing,1599,RetailMart Sector 17,Chandigarh,North,3198.0
8,10,110,10,1.0,2024-01-09,2499.0,Bluetooth Speaker,Electronics,2499,RetailMart Linking Road,Mumbai,West,2499.0
9,11,111,3,3.0,2024-01-10,1350.0,Basmati Rice,Grocery,450,RetailMart Indiranagar,Bangalore,South,1350.0
